# 01 · Exploring ZAP project data

**Goal:** understand the Zoning Application Portal (ZAP) project dataset before writing the Phase 1 fetcher.
What does one row represent? Which columns are reliable? What quirks will the silver model need to handle?

**Dataset:** [ZAP Project Data](https://data.cityofnewyork.us/d/hgx4-8ukb) · ID `hgx4-8ukb` · published by NYC Department of City Planning

**How this notebook works**

1. *Setup* and *Load* are done for you. Run them once; the data is cached to `data/explore/` so reruns don't call the API.
2. *Shape* shows two example queries.
3. *Your questions* has prompts with hints. Write the SQL yourself, then note what you found.
4. *Findings* is where the answers go. They become `docs/sources/zap.md`.

To refetch fresh data, delete `data/explore/zap_projects.json` and rerun.

## Setup

In [49]:
import json
import os
from pathlib import Path

import duckdb
import requests
import pandas as pd
import numpy

# Find the repo root whether Jupyter was started from the repo or from notebooks/
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE = REPO / "data" / "explore" / "zap_projects.json"

BASE_URL = "https://data.cityofnewyork.us/resource/hgx4-8ukb.json"


def load_env(path: Path) -> None:
    """Read KEY=value lines from .env into environment variables (skips comments and blanks)."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


load_env(REPO / ".env")
TOKEN = os.environ.get("SOCRATA_APP_TOKEN")
HEADERS = {"X-App-Token": TOKEN} if TOKEN else {}
print("Repo:", REPO)
print("Using app token:", bool(HEADERS))

Repo: /Users/liz/Documents/nyc-land-use
Using app token: True


## Load

Fetches every row in pages of 10,000, ordered by `project_id` so pages never overlap or skip rows.
This is a preview of the pagination your Phase 1 fetcher will do properly, with tests.

In [2]:
def fetch_all(page_size: int = 10_000) -> list[dict]:
    rows, offset = [], 0
    while True:
        params = {"$limit": page_size, "$offset": offset, "$order": "project_id"}
        resp = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=60)
        resp.raise_for_status()
        page = resp.json()
        rows.extend(page)
        print(f"  fetched {len(page):,} rows at offset {offset:,}")
        if len(page) < page_size:
            return rows
        offset += page_size


if CACHE.exists():
    print(f"Using cached file {CACHE.relative_to(REPO)}")
else:
    print("Fetching from the API...")
    data = fetch_all()
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    CACHE.write_text(json.dumps(data))
    print(f"Saved {len(data):,} rows to {CACHE.relative_to(REPO)}")

Fetching from the API...
  fetched 10,000 rows at offset 0
  fetched 10,000 rows at offset 10,000
  fetched 10,000 rows at offset 20,000
  fetched 2,964 rows at offset 30,000
Saved 32,964 rows to data/explore/zap_projects.json


Load the cached JSON into an in-memory DuckDB database. `sample_size=-1` tells DuckDB to scan
every row when guessing column types, which matters here: the API **leaves a field out entirely when
it's empty**, so the first few rows don't show every column.

In [3]:
con = duckdb.connect()  # in-memory database: nothing is written to disk
con.sql(f"""
    create or replace table zap as
    select * from read_json('{CACHE.as_posix()}', sample_size = -1)
""")
con.sql("select count(*) as rows from zap")

┌───────┐
│ rows  │
│ int64 │
├───────┤
│ 32964 │
└───────┘

**Check against the source.** If these two numbers differ, something went wrong in the fetch.
(When this notebook was written, the API reported 32,964 rows.)

In [4]:
def api_count() -> int | None:
    try:
        resp = requests.get(BASE_URL, params={"$select": "count(*)"}, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        return int(resp.json()[0]["count"])
    except requests.RequestException as err:
        print("Couldn't reach the API:", err)
        return None


local = con.sql("select count(*) from zap").fetchone()[0]
remote = api_count()
print(f"local: {local:,}   api: {remote if remote is None else f'{remote:,}'}")

local: 32,964   api: 32,964


## Shape

Two quick ways to get oriented. `describe` lists columns and the types DuckDB inferred;
`summarize` gives per-column stats including `null_percentage`.

In [50]:
con.sql("describe zap").df()

,column_name,column_type,null,key,default,extra
0,project_id,VARCHAR,YES,None,None,None
1,project_name,VARCHAR,YES,None,None,None
2,project_brief,VARCHAR,YES,None,None,None
3,project_status,VARCHAR,YES,None,None,None
4,public_status,VARCHAR,YES,None,None,None
5,ulurp_non,VARCHAR,YES,None,None,None
6,primary_applicant,VARCHAR,YES,None,None,None
7,applicant_type,VARCHAR,YES,None,None,None
8,borough,VARCHAR,YES,None,None,None
9,community_district,VARCHAR,YES,None,None,None


In [51]:
con.sql("""
    select column_name, column_type, approx_unique, null_percentage
    from (summarize zap)
    order by null_percentage desc
""").df()

,column_name,column_type,approx_unique,null_percentage
0,mih_mapped_no_res,VARCHAR,2,99.72
1,noticed_date,VARCHAR,146,98.56
2,current_envmilestone_date,VARCHAR,454,98.52
3,current_envmilestone,VARCHAR,26,98.52
4,approval_date,VARCHAR,505,97.11
5,eas_eis,VARCHAR,3,96.92
6,app_filed_date,VARCHAR,1095,95.63
7,mih_workforce,VARCHAR,2,95.46
8,actions,VARCHAR,224,95.42
9,mih_option2,VARCHAR,2,95.29


Look at a few full rows, newest first. (`order by ... nulls last` keeps rows with no filing date out of the way.)

In [ ]:
con.sql("select * from zap order by app_filed_date desc nulls last limit 5")

## Your questions

Write a query in each empty cell. Hints are in the comments; delete them once you're done.
After each one, add a line to *Findings* below.

### Q1. Is `project_id` unique?
If it is, it can be the primary key in silver. If not, you'll need to decide which row wins.

In [14]:
# Hint: compare count(*) with count(distinct project_id).
# Then: are there any rows where project_id is null?

total_rows = con.sql("select count(*) from zap").fetchone()[0]
distinct_rows = con.sql("select count(distinct project_id) from zap").fetchone()[0]

total_rows == distinct_rows

True

In [52]:
# Check for nulls
# Conclusion: No nulls in the project ID field
con.sql("select * FROM zap WHERE project_id IS NULL limit 10").df()

,project_id,project_name,project_brief,project_status,public_status,ulurp_non,primary_applicant,applicant_type,borough,community_district,...,current_envmilestone,current_envmilestone_date,noticed_date,approval_date,mih_flag,mih_option1,mih_option2,mih_workforce,mih_deepaffordability,mih_mapped_no_res


### Q2. What values do the categorical columns take?
Check `ulurp_non`, `project_status`, `public_status`, `borough`, `applicant_type`.
Watch for inconsistent spelling or capitalization, and nulls.

Conclusion: The categorical variables are all quite clean.

In [30]:
# Hint: select <column>, count(*) as n from zap group by all order by n desc
con.sql("select borough, count(*) as n from zap group by all order by n desc")

┌───────────────┬───────┐
│    borough    │   n   │
│    varchar    │ int64 │
├───────────────┼───────┤
│ Manhattan     │ 11614 │
│ Staten Island │  8395 │
│ Brooklyn      │  5053 │
│ Queens        │  3686 │
│ Bronx         │  3219 │
│ Citywide      │   997 │
└───────────────┴───────┘

### Q3. How does coverage change over time?
Projects go back to the 1970s. Do older records have the same fields filled in as newer ones?

In [53]:
# Hint: group by decade of app_filed_date, e.g. (year(app_filed_date::timestamp) // 10) * 10,
# (the ::timestamp cast is needed if DuckDB read the dates as text: see Q4)
# and compute the share of rows with cc_district, certified_referred, approval_date not null.
# avg((cc_district is not null)::int) gives the share as a number between 0 and 1.
# If app_filed_date is null for old records, try certified_referred instead.

con.sql("""
    select 
        (year(certified_referred::timestamp) // 10) * 10 as decade,
        count(*) as total_projects,
        avg((app_filed_date is not null::int)) as share_app_file_date,
        avg((cc_district is not null)::int) as share_cc_district,
        avg((certified_referred is not null)::int) as share_certified_referred,
        avg((approval_date is not null)::int) as share_approval_date
    from zap
    group by decade
    order by decade
""").df()


# Notes: Only 324 projects from 2010s, 1115 from 2020s, 31525 are NULL. Maybe they don't have the date field?.
# Only 3% have an approval date in the 2010s, .5% in the NULL fields, andf 69% in the 2020s.
# CC district is similarly mostly NULL in the NULL decades, and only 44% in the 2010s, going up to 94% in the 2020s

# Certified referred date is in records going back to the 1970s. cc district is mostly null until the 2020s.

,decade,total_projects,share_app_file_date,share_cc_district,share_certified_referred,share_approval_date
0,1970,2096,0.000000,0.000000,1.0,0.000000
1,1980,11362,0.000000,0.000088,1.0,0.000000
2,1990,7016,0.000000,0.000143,1.0,0.000000
3,2000,5212,0.000000,0.003262,1.0,0.000000
4,2010,5123,0.040406,0.052313,1.0,0.002342
5,2020,1242,0.803543,0.914654,1.0,0.747987
6,<NA>,913,0.256298,0.302300,0.0,0.013143


### Q4. What do the date columns look like?
Did DuckDB read them as timestamps or as text? Are there impossible dates (before 1960, in the future)?

In [54]:
# Hint: check column_type in the `describe` output above. The API sends every value as
# text, so dates (and cc_district) may be VARCHAR. Cast with ::timestamp, then take min()
# and max() of each *_date column and certified_referred.

con.sql("select certified_referred from zap limit 10")

con.sql("""
    select 
        (max(certified_referred::timestamp)) as max_certified_referred,
        (min(certified_referred::timestamp)) as min_certified_referred,
        (max(current_envmilestone_date::timestamp))as max_current_envmilestone_date,
        (min(current_envmilestone_date::timestamp)) as min_current_envmilestone_date,
        (max(noticed_date::timestamp))as max_noticed_date,
        (min(noticed_date::timestamp)) as min_noticed_date,
        count(*) as total_projects
    from zap
""").df()

,max_certified_referred,min_certified_referred,max_current_envmilestone_date,min_current_envmilestone_date,max_noticed_date,min_noticed_date,total_projects
0,2026-07-13,1970-04-21,2026-07-02,2017-04-08,2026-06-26,2020-08-14,32964


### Q5. Do the dates make sense in order?
A project should be filed before it's certified, and certified before it's approved or completed.

In [61]:
# Hint: count rows where completed_date < app_filed_date, or approval_date < certified_referred.
# Look at a few: is it a data error, or does the field mean something other than you assumed?

con.sql("""select 
                project_id, completed_date, app_filed_date, approval_date, certified_referred,
                from zap 
                where completed_date < app_filed_date OR approval_date < certified_referred""").df()

,project_id,completed_date,app_filed_date,approval_date,certified_referred
0,2018K0358,2019-02-15T00:00:00.000,2021-07-01T00:00:00.000,NaN,2018-08-06T00:00:00.000
1,2020M0301,2021-07-21T00:00:00.000,2020-12-23T00:00:00.000,2021-07-07T00:00:00.000,2021-07-21T00:00:00.000
2,2022K0297,2022-03-04T00:00:00.000,2022-02-17T00:00:00.000,2022-03-02T00:00:00.000,2022-03-09T00:00:00.000
3,2022K0299,2022-03-07T00:00:00.000,2022-02-18T00:00:00.000,2022-03-02T00:00:00.000,2022-03-09T00:00:00.000
4,2022K0303,2022-03-07T00:00:00.000,2022-02-18T00:00:00.000,2022-03-07T00:00:00.000,2022-04-06T00:00:00.000
5,2022K0307,2022-04-12T00:00:00.000,2022-02-22T00:00:00.000,2022-03-02T00:00:00.000,2022-04-06T00:00:00.000
6,2022K0324,2022-03-07T00:00:00.000,2022-02-17T00:00:00.000,2022-03-02T00:00:00.000,2022-03-09T00:00:00.000
7,2022K0348,2022-03-17T00:00:00.000,2022-03-08T00:00:00.000,2022-03-11T00:00:00.000,2022-04-06T00:00:00.000
8,2022K0351,2022-04-12T00:00:00.000,2022-03-18T00:00:00.000,2022-04-04T00:00:00.000,2022-04-06T00:00:00.000
9,2022K0374,2022-04-12T00:00:00.000,2022-03-29T00:00:00.000,2022-04-05T00:00:00.000,2022-04-06T00:00:00.000


### Q6. What does `ulurp_numbers` look like?
This field will probably link projects to City Council records in Phase 8.
Is it one number per project or several? How are multiple values separated?

In [67]:
# Hint: select ulurp_numbers from zap where ulurp_numbers is not null limit 20
# Then count how many contain a separator, e.g. where ulurp_numbers like '%;%'

con.sql("select actions from zap where ulurp_numbers is not null limit 20").df()

# Several values separated by a semicolon

,actions
0,ZM; ZR
1,NaN
2,ZA
3,NaN
4,NaN
5,NaN
6,ZM; ZR; MM; MM; PP; HA
7,NaN
8,NaN
9,ZM; ZR


### Q7. How could you identify housing-related projects?
This decides which projects the site focuses on. Candidates: the `mih_*` flags, codes in `actions`,
keywords in `project_brief`. How many projects does each approach catch, and how much do they overlap?

In [66]:
# Hint: start with count(*) filter (where mih_flag is not null), then look at what's in `actions`.

con.sql("select actions from zap where mih_flag is not null limit 20")

# actions includes HO

┌────────────────────────────────┐
│            actions             │
│            varchar             │
├────────────────────────────────┤
│ ZM; ZR                         │
│ NULL                           │
│ NULL                           │
│ ZM; ZR; MM; MM; PP; HA         │
│ ZM; ZR                         │
│ ZM; ZR                         │
│ NULL                           │
│ ZM; ZR                         │
│ ZM; ZR                         │
│ ZM; ZR                         │
│ NULL                           │
│ HA                             │
│ ZM; ZR                         │
│ ZM                             │
│ ZM; ZR; ZS; ZC; ZA; ZA; LD; ZA │
│ ZM; ZR                         │
│ ZM; ZR                         │
│ ZC; LD                         │
│ NULL                           │
│ NULL                           │
└────────────────────────────────┘
             20 rows            

### Q8. Anything else that surprised you?
Free space for whatever caught your eye.

## Findings

Fill these in as you go, then copy them into `docs/sources/zap.md`.

- **Rows / grain:** one row per ...
- **Primary key:**
- **Categorical values and quirks:** The categorical variables are all quite clean in terms of consistent spelling/capitalization. There are 2118 NULLS in applicant_type, 2373 in ulurp_non, 1 in public_status
- **Coverage over time:**
- **Date fields:**
- **`ulurp_numbers` format:**
- **Identifying housing projects:**
- **Update cadence:** `rowsUpdatedAt` in the [dataset metadata](https://data.cityofnewyork.us/api/views/hgx4-8ukb.json)
- **Decisions for the silver model:**